# 🚀 Hybrid RAG System with Cross-Encoder Reranking & Evaluation

An interview-grade, production-style Retrieval-Augmented Generation (RAG) system designed to eliminate retrieval failure modes on enterprise documents.

---

## 🏗️ Architecture Overview

```text
                               PDF Documents
                                     │
                                     ▼
                            [ 1. PDF Loading ]
                        (Extract text + page metadata)
                                     │
                                     ▼
                           [ 2. Text Chunking ]
                   (Split into small, coherent passages)
                                     │
                    ┌────────────────┴────────────────┐
                    ▼                                 ▼
         [ 3A. Dense Retrieval ]             [ 3B. BM25 Retrieval ]
       • Sentence/Nomic Embeddings         • Term frequency / IDF
       • Stored in Pinecone Vector DB      • Exact keywords, codes, names
       • Finds: 'hardware vs services'     • Finds: 'Q1 2024', '$119.58B'
                    │                                 │
                    └────────────────┬────────────────┘
                                     ▼
                      [ 4. Reciprocal Rank Fusion (RRF) ]
                       Merge & deduplicate candidate pools
                                     │
                                     ▼
                     [ 5. Cross-Encoder Reranking ]
                      (FlashRank / Cross-Encoder)
                  Jointly scores (Query, Chunk) pairs
                                     │
                                     ▼
                         [ Top-K Selected Chunks ]
                                     │
                                     ▼
                          [ 6. Prompt + Context ]
                                     │
                                     ▼
                          [ 7. Local Ollama LLM ]
                               (llama3.2:3b)
                                     │
                                     ▼
                      [ 8. Grounded Answer + Sources ]
                                     │
                    ┌────────────────┴────────────────┐
                    ▼                                 ▼
         [ 9A. Retrieval Precision@K ]      [ 9B. Faithfulness Eval ]
          Did we retrieve the right pages?   Is the answer backed by text?
```

---

## Section 1: Why Hybrid RAG? (The Problem We Are Solving)

Most beginner RAG tutorials only implement **Dense Vector Search** (`vectorstore.similarity_search()`). While embeddings are great at finding conceptual similarity, they fail in two common real-world scenarios:

1. **Exact Numbers & Financial Codes:** Dense embeddings compress an entire 500-token paragraph into 768 floating-point numbers. In this compression, specific numbers (e.g., `$119.58 billion` vs `$117.15 billion`) or fiscal quarters (`Q1 2024` vs `Q1 2023`) get blurred into general "financial revenue" vectors.
2. **Specific Term Matching:** Alphanumeric codes, rare product SKUs, or section titles often have poor representation in generic embedding spaces.

### The Solution:
* **BM25 (Sparse Lexical Search):** Directly scores exact word matches using Term Frequency (TF) and Inverse Document Frequency (IDF). It never misses an exact keyword.
* **Dense Search:** Understands synonyms and high-level concepts.
* **Reciprocal Rank Fusion (RRF):** Combines the ranks from both methods safely.
* **Cross-Encoder Reranking:** Re-scores the top candidates with full attention before passing them to the LLM.

## Section 2: Environment & Dependency Verification

In this section, we load our environment variables, verify our local Ollama instance (`llama3.2:3b`), and test our connection to the Pinecone cloud vector database.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "apple-hybrid-rag")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")

print("✅ Environment configuration loaded:")
print(f"  • Pinecone Index Name: {PINECONE_INDEX_NAME}")
print(f"  • Pinecone API Key set: {'Yes (length ' + str(len(PINECONE_API_KEY)) + ')' if PINECONE_API_KEY else '❌ NOT SET'}")
print(f"  • Ollama Base URL: {OLLAMA_BASE_URL}")
print(f"  • Ollama Model: {OLLAMA_MODEL}")

In [ ]:
# Verify Ollama Local LLM Connection
from langchain_ollama import ChatOllama

try:
    llm = ChatOllama(
        model=OLLAMA_MODEL,
        temperature=0.2,
        base_url=OLLAMA_BASE_URL
    )
    test_response = llm.invoke("Respond with 'Ollama is ready!' in 3 words.")
    print("✅ Ollama Connection Successful!")
    print("  Model Response:", test_response.content.strip())
except Exception as e:
    print("❌ Ollama connection failed. Make sure Ollama is running (`ollama serve`):")
    print("  Error:", e)

In [1]:
# Verify Pinecone Connection
from pinecone import Pinecone

try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
    indexes = pc.list_indexes().names()
    print("✅ Pinecone Connection Successful!")
    print(f"  Existing indexes in your account: {indexes}")
except Exception as e:
    print("❌ Pinecone connection failed. Check your PINECONE_API_KEY in .env:")
    print("  Error:", e)

ModuleNotFoundError: No module named 'pinecone'